# HubbardBath 02 — Gibbs states and thermal observables

This notebook moves from the zero-temperature spectrum to **finite-temperature physics** in the half-filled two-site Fermi–Hubbard model.

We work in the canonical ensemble with exactly two electrons and construct

$$\rho_\beta = \frac{e^{-\beta H}}{\mathrm{Tr}(e^{-\beta H})}.$$

The experiment asks how interaction strength `U/t` and inverse temperature `βt` affect energy, entropy, double occupancy, and nearest-neighbour spin correlations.


In [ ]:
from pathlib import Path
import sys
import numpy as np
import matplotlib.pyplot as plt

candidates = [Path.cwd(), Path.cwd().parent, Path.cwd() / 'HubbardBath']
for candidate in candidates:
    if (candidate / 'src' / 'hubbard.py').exists():
        repo_root = candidate.resolve()
        break
else:
    raise FileNotFoundError('Run this notebook from the HubbardBath repository.')

if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from src.hubbard import project_to_particle_sector, two_site_hubbard_hamiltonian
from src.thermal import (
    average_double_occupancy_operator,
    canonical_thermal_observables,
    gibbs_state,
    nearest_neighbor_spin_z_correlation_operator,
    project_operator_to_sector,
)


## 1. Build the fixed-particle-number thermal problem

Half filling for two sites means exactly two electrons. We project both the Hamiltonian and observables into this six-dimensional sector before constructing the Gibbs state.


In [ ]:
U = 4.0
beta = 1.0

H_full = two_site_hubbard_hamiltonian(t=1.0, U=U)
H_half, _ = project_to_particle_sector(H_full, num_modes=4, particle_number=2)

D_half = project_operator_to_sector(
    average_double_occupancy_operator(), 4, 2
)
Czz_half = project_operator_to_sector(
    nearest_neighbor_spin_z_correlation_operator(), 4, 2
)

rho = gibbs_state(H_half, beta)
print('Trace:', np.trace(rho))
print('Minimum eigenvalue:', np.linalg.eigvalsh(rho).min())
print('Hermiticity error:', np.linalg.norm(rho - rho.conj().T))


## 2. Thermal observables at one point

- **Energy** probes the thermal occupation of many-body eigenstates.
- **Entropy** measures mixedness of the thermal state.
- **Double occupancy** measures how often opposite-spin electrons share a site.
- **Spin-z correlation** probes whether neighbouring spins tend to align or anti-align along z.


In [ ]:
obs = canonical_thermal_observables(H_half, beta, D_half, Czz_half)
for key, value in obs.items():
    print(f'{key:22s}: {value:.6f}')


## 3. Sweep interaction strength at several temperatures

We now ask how the observables change as the competition between hopping and repulsion changes.


In [ ]:
U_values = np.linspace(0.0, 8.0, 81)
beta_values = [0.25, 0.5, 1.0, 2.0, 5.0]

results = {beta: {k: [] for k in ['energy','entropy','double_occupancy','spin_z_correlation']} for beta in beta_values}

for U_value in U_values:
    H_full = two_site_hubbard_hamiltonian(t=1.0, U=float(U_value))
    H_half, _ = project_to_particle_sector(H_full, 4, 2)
    for beta_value in beta_values:
        obs = canonical_thermal_observables(H_half, beta_value, D_half, Czz_half)
        for key, value in obs.items():
            results[beta_value][key].append(value)


In [ ]:
plt.figure(figsize=(7, 4.5))
for beta_value in beta_values:
    plt.plot(U_values, results[beta_value]['double_occupancy'], label=fr'$\beta t={beta_value}$')
plt.xlabel(r'Interaction strength $U/t$')
plt.ylabel('Average double occupancy')
plt.title('Interaction suppresses double occupancy')
plt.legend()
plt.grid(alpha=0.2)
plt.tight_layout()
plt.show()


In [ ]:
plt.figure(figsize=(7, 4.5))
for beta_value in beta_values:
    plt.plot(U_values, results[beta_value]['spin_z_correlation'], label=fr'$\beta t={beta_value}$')
plt.xlabel(r'Interaction strength $U/t$')
plt.ylabel(r'$\langle S^z_0 S^z_1 \rangle$')
plt.title('Nearest-neighbour spin correlation')
plt.legend()
plt.grid(alpha=0.2)
plt.tight_layout()
plt.show()


## 4. Two-dimensional thermal map

The full sweep is over both `U/t` and `βt`. These maps will later become the baseline against which open-system relaxation is compared.


In [ ]:
U_grid = np.linspace(0.0, 8.0, 41)
beta_grid = np.linspace(0.0, 5.0, 41)
double_map = np.zeros((len(beta_grid), len(U_grid)))
spin_map = np.zeros_like(double_map)

for i, beta_value in enumerate(beta_grid):
    for j, U_value in enumerate(U_grid):
        H_full = two_site_hubbard_hamiltonian(t=1.0, U=float(U_value))
        H_half, _ = project_to_particle_sector(H_full, 4, 2)
        obs = canonical_thermal_observables(H_half, float(beta_value), D_half, Czz_half)
        double_map[i, j] = obs['double_occupancy']
        spin_map[i, j] = obs['spin_z_correlation']


In [ ]:
plt.figure(figsize=(7, 4.8))
im = plt.imshow(double_map, origin='lower', aspect='auto', extent=[0,8,0,5])
plt.xlabel(r'$U/t$')
plt.ylabel(r'$\beta t$')
plt.title('Average double occupancy')
plt.colorbar(im)
plt.tight_layout()
plt.show()


## What Milestone 02 establishes

We now have an exact finite-temperature baseline for the same interacting fermionic model used in Milestone 01. The next step is to **leave equilibrium**: choose a non-equilibrium initial state, couple the system to a thermal environment, and ask how quickly the Lindblad dynamics recover these Gibbs-state observables.
